# Transcripción de clases con Whisper `large-v3`

Este cuaderno transcribe clases largas con alta calidad, vocabulario técnico y guardado automático del progreso.

**Antes de comenzar:** en Google Colab selecciona:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

La disponibilidad de GPU gratuita depende de Google Colab.


In [ ]:
# Instala el motor de transcripción
!pip -q install -U faster-whisper tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import torch
from faster_whisper import WhisperModel
from tqdm.auto import tqdm

print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. Configura la ruta y el archivo

En la siguiente celda debes cambiar únicamente los valores señalados con:

`# <<< CAMBIAR AQUÍ >>>`

Ejemplo de estructura en Drive:

`Mi unidad / Transcripciones / NOMBRE_DE_LA_CARPETA / archivo.mp4`

Puedes usar `.mp4`, `.mp3`, `.wav`, `.m4a` u otro formato compatible con Whisper.


In [ ]:
# ============================================================
# CONFIGURACIÓN PRINCIPAL
# CAMBIA SOLO LAS DOS LÍNEAS MARCADAS
# ============================================================

# <<< CAMBIAR AQUÍ >>>
# Escribe la ruta de la carpeta de Google Drive donde está tu audio o video.
# Conserva siempre al inicio: /content/drive/MyDrive/
CARPETA_DRIVE = Path(
    "/content/drive/MyDrive/Transcripciones/NOMBRE_DE_LA_CARPETA"
)

# <<< CAMBIAR AQUÍ >>>
# Escribe exactamente el nombre del archivo, incluida su extensión.
NOMBRE_ARCHIVO_ENTRADA = "NOMBRE_DEL_ARCHIVO.mp4"

# ============================================================
# DESDE AQUÍ NO ES NECESARIO CAMBIAR NADA
# ============================================================

archivo_entrada = CARPETA_DRIVE / NOMBRE_ARCHIVO_ENTRADA
nombre_base = archivo_entrada.stem

archivo_salida = (
    CARPETA_DRIVE / f"{nombre_base}_Transcripcion_Fase_1.txt"
)
archivo_progreso = (
    CARPETA_DRIVE / f"{nombre_base}_progreso.jsonl"
)

# Crea la carpeta si todavía no existe.
CARPETA_DRIVE.mkdir(parents=True, exist_ok=True)

if not archivo_entrada.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{archivo_entrada}\n\n"
        "Verifica CARPETA_DRIVE y NOMBRE_ARCHIVO_ENTRADA."
    )

print("Entrada:", archivo_entrada)
print("Salida:", archivo_salida)
print("Progreso:", archivo_progreso)


## 2. Verifica los archivos de la carpeta

Esta celda es opcional. Sirve para comprobar que Colab está viendo correctamente el contenido de la carpeta configurada.


In [ ]:
if CARPETA_DRIVE.exists():
    print(f"Archivos en: {CARPETA_DRIVE}\n")
    for item in sorted(os.listdir(CARPETA_DRIVE)):
        print("•", item)
else:
    print("La carpeta no existe:", CARPETA_DRIVE)


## 3. Contexto técnico para Whisper

Este bloque ayuda a Whisper a reconocer términos propios de la asignatura.

Si el cuaderno se va a usar para **otra materia**, puedes cambiar el texto de `PROMPT_INICIAL_CURSO` y `VOCABULARIO_TECNICO_CURSO`. No necesitas modificarlo si vas a transcribir clases de Arquitectura de Nube o temas similares.


In [ ]:
PROMPT_INICIAL_CURSO = """
Clase universitaria en español de Ingeniería de Sistemas sobre
arquitecturas de nube y sistemas distribuidos.

Transcribir literalmente, sin resumir. Conservar correctamente los
términos técnicos, siglas, protocolos, comandos y palabras en inglés.
""".strip()


VOCABULARIO_TECNICO_CURSO = """
AWS, cloud computing, sistemas distribuidos, arquitectura de software,
cliente-servidor, middleware, microservicios, máquina virtual,
virtualización, Docker, Kubernetes, API REST, servidor, cliente,
host, modelo OSI, TCP/IP, dirección IP, IPv4, IPv6, CIDR, subnetting,
NAT, gateway, DNS, DHCP, router, switch, firewall, socket, puerto,
TCP, UDP, HTTP, HTTPS, FTP, SSH, Telnet, SMTP, Linux, Windows,
PowerShell, Git, GitHub, ipconfig, netstat, ping, traceroute, nmap.
""".strip()


## 4. Funciones para guardar y recuperar el progreso

No necesitas modificar esta celda.


In [ ]:
def cargar_progreso_guardado():
    segmentos_guardados = []
    transcripcion_finalizada = False
    registros_validos = []

    if not archivo_progreso.exists():
        return segmentos_guardados, transcripcion_finalizada

    with open(archivo_progreso, "r", encoding="utf-8") as archivo:
        for linea in archivo:
            linea = linea.strip()

            if not linea:
                continue

            try:
                registro = json.loads(linea)
            except json.JSONDecodeError:
                # Si la última línea quedó incompleta por una interrupción,
                # se conserva todo lo válido hasta ese punto.
                break

            registros_validos.append(registro)

            if registro.get("tipo") == "segmento":
                segmentos_guardados.append(registro)
            elif registro.get("tipo") == "completado":
                transcripcion_finalizada = True

    # Reescribe el archivo únicamente con registros válidos.
    with open(archivo_progreso, "w", encoding="utf-8") as archivo:
        for registro in registros_validos:
            archivo.write(
                json.dumps(registro, ensure_ascii=False) + "\n"
            )

    return segmentos_guardados, transcripcion_finalizada


def reconstruir_transcripcion(segmentos_guardados):
    with open(archivo_salida, "w", encoding="utf-8") as archivo:
        for segmento in segmentos_guardados:
            texto = segmento.get("texto", "").strip()

            if texto:
                archivo.write(texto + "\n")


def guardar_registro(archivo, registro):
    archivo.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )
    archivo.flush()


## 5. Ejecuta la transcripción

Esta celda:

- carga Whisper `large-v3`;
- continúa desde el último segmento guardado si hubo una interrupción;
- guarda la transcripción en `.txt`;
- guarda un archivo `.jsonl` con el progreso;
- permite volver a ejecutar la misma celda para continuar.


In [ ]:
segmentos_guardados, transcripcion_finalizada = cargar_progreso_guardado()
reconstruir_transcripcion(segmentos_guardados)

if transcripcion_finalizada:
    print("✅ Esta clase ya fue transcrita completamente.")
    print("Archivo:", archivo_salida)

else:
    segundo_reanudacion = (
        float(segmentos_guardados[-1]["fin"])
        if segmentos_guardados
        else 0.0
    )

    print(
        f"Reanudación desde: {segundo_reanudacion:.2f} segundos"
    )

    prompt_completo = PROMPT_INICIAL_CURSO

    dispositivo = (
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    tipo_calculo = (
        "float16" if dispositivo == "cuda" else "int8"
    )

    if dispositivo == "cpu":
        print(
            "⚠️ No hay GPU. large-v3 funcionará, "
            "pero será mucho más lento."
        )

    print(
        f"Cargando large-v3 en {dispositivo} "
        f"con {tipo_calculo}..."
    )

    modelo = WhisperModel(
        "large-v3",
        device=dispositivo,
        compute_type=tipo_calculo
    )

    segmentos, informacion = modelo.transcribe(
        str(archivo_entrada),

        language="es",
        task="transcribe",

        # Alta precisión
        beam_size=5,
        temperature=0.0,

        # Contexto técnico fijo
        initial_prompt=prompt_completo,
        hotwords=VOCABULARIO_TECNICO_CURSO,

        # Evita que el prompt crezca con cada fragmento
        condition_on_previous_text=False,

        # Cantidad suficiente para cada fragmento de audio
        max_new_tokens=128,

        # Continúa desde el último punto guardado
        clip_timestamps=f"{segundo_reanudacion:.3f}",

        vad_filter=True,
        vad_parameters={
            "min_silence_duration_ms": 1500
        },

        word_timestamps=False
    )

    duracion_total = informacion.duration
    ultimo_segundo = segundo_reanudacion

    try:
        with open(
            archivo_progreso, "a", encoding="utf-8"
        ) as progreso, open(
            archivo_salida, "a", encoding="utf-8"
        ) as salida, tqdm(
            total=duracion_total,
            initial=min(
                segundo_reanudacion,
                duracion_total
            ),
            desc="Transcribiendo",
            unit="s"
        ) as barra:

            for segmento in segmentos:
                texto = segmento.text.strip()

                registro = {
                    "tipo": "segmento",
                    "inicio": round(
                        segmento.start, 3
                    ),
                    "fin": round(
                        segmento.end, 3
                    ),
                    "texto": texto
                }

                guardar_registro(
                    progreso,
                    registro
                )

                if texto:
                    salida.write(texto + "\n")
                    salida.flush()

                avance = (
                    segmento.end - ultimo_segundo
                )

                if avance > 0:
                    barra.update(avance)

                ultimo_segundo = segmento.end

            guardar_registro(
                progreso,
                {
                    "tipo": "completado",
                    "fin": duracion_total
                }
            )

    except KeyboardInterrupt:
        print(
            "\n⚠️ Transcripción interrumpida."
        )
        print(
            "Progreso guardado hasta "
            f"{ultimo_segundo:.2f} segundos."
        )
        print(
            "Vuelve a ejecutar esta celda "
            "para continuar."
        )

    else:
        print("\n✅ Transcripción finalizada.")
        print("Archivo:", archivo_salida)


## Resultado

Al terminar, en la misma carpeta de Google Drive aparecerán:

- `NOMBRE_DEL_ARCHIVO_Transcripcion_Fase_1.txt` → transcripción final.
- `NOMBRE_DEL_ARCHIVO_progreso.jsonl` → registro del progreso.

### Para transcribir otra clase

Cambia únicamente:

1. `CARPETA_DRIVE`, si el archivo está en otra carpeta.
2. `NOMBRE_ARCHIVO_ENTRADA`.

### Para comenzar nuevamente una transcripción desde cero

Elimina de Google Drive los dos archivos generados para esa clase:

- `_Transcripcion_Fase_1.txt`
- `_progreso.jsonl`

y vuelve a ejecutar el cuaderno.
